# Interest Rate Mathematics

A from-scratch exploration of interest rate fundamentals: simple and compound interest, continuous compounding, rate conversions, discount factors, forward rates, and day-count conventions — all built with NumPy.This notebook covers the complete CFA Level 1 interest rate mathematics curriculum. Every concept is explained intuitively before being implemented — you'll understand the "why" before the "how."

> **Prerequisites:** Basic algebra and an understanding of what interest rates are. No prior knowledge of finance mathematics is assumed.


## 1. Motivation

Interest rates are the **foundation of quantitative finance**. Every bond price, swap valuation, mortgage payment, and derivatives model ultimately rests on an understanding of how money grows over time.

The time value of money is captured by a single idea: **a dollar today is worth more than a dollar tomorrow**, because today's dollar can be invested to earn interest. This is not just a theoretical abstraction — it is the reason banks pay you to deposit money, the reason loans cost more than the principal, and the reason retirement planning works at all.

### What We Will Cover

This notebook builds the complete interest rate toolkit from the ground up:

| Topic | What It Answers | Real-World Application |
|-------|----------------|------------------------|
| Simple Interest | How does money grow linearly? | T-bills, short-term bank deposits |
| Compound Interest | How does money grow exponentially? | Savings accounts, bonds, loans |
| Continuous Compounding | What is the theoretical limit of compounding? | Derivatives pricing (Black-Scholes) |
| Rate Conversions | How do I compare rates quoted differently? | Comparing a bond yield to a savings rate |
| Discount Factors | How much is future money worth today? | Valuing any financial asset |
| Forward Rates | What does the market expect future rates to be? | Hedging, rate expectations |
| Day-Count Conventions | How do I calculate exact interest amounts? | Bond accrued interest, swap payments |

### Why This Matters for Your Career

Whether you are preparing for the CFA exam, working at a bank, or managing your own investments, interest rate math is the language of finance. A portfolio manager who misunderstands compounding conventions can misprice a bond. A risk manager who confuses nominal and effective rates can miscalculate hedging ratios. The concepts in this notebook are not optional — they are the price of admission.

> **CFA Exam Tip:** The CFA Level I exam heavily tests time value of money (TVM) concepts. You should be able to convert between different compounding conventions, compute discount factors, and derive forward rates without a calculator showing intermediate steps. Expect 5-10 questions on these topics.### What You Will Learn

| Topic | Key Question It Answers |
|:------|:----------------------|
| Simple interest | How does money grow linearly over time? |
| Compound interest | What is "interest on interest" and why does it matter so much? |
| Continuous compounding | What happens in the limit of infinite compounding? |
| Rate conversions | How do I compare rates quoted on different bases? |
| Discount factors | How much is a future dollar worth today? |
| Forward rates | What rate can I lock in today for borrowing in the future? |
| Day count conventions | Why do different markets count days differently? |

> **Key Concept:** Understanding interest rate mathematics is essential because virtually every financial calculation — bond pricing, loan payments, project valuation, derivatives pricing — reduces to discounting cash flows. If you master this notebook, you have the tools to tackle all of them.
### A Note on Notation

Different markets and textbooks use different notation for the same concepts. Here's a quick reference:

| This notebook | Also called | Symbol |
|:---|:---|:---|
| Nominal rate | Stated rate, APR, quoted rate | $r_{\text{nom}}$ |
| Effective rate | EAR, APY | $r_{\text{eff}}$ |
| Continuous rate | Log rate, force of interest | $r_c$ or $\delta$ |
| Discount factor | Present value factor | $d(t)$ or $P(0,t)$ |
| Forward rate | Implied forward | $f(t_1, t_2)$ |


## 2. SetupWe implement everything from scratch using NumPy and SciPy — no financial libraries. This lets you see exactly how each formula is computed and verify results by hand.


In [ ]:
%matplotlib inline
import numpy as np
from scipy import linalg, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

## 3. Simple Interest

### The Idea in Plain English

Simple interest is the most basic way to calculate the cost of borrowing (or the reward for lending). Think of it like renting money: if you borrow $10,000 at 5% per year, you owe a flat $500 per year in "rent" for that money — regardless of how long you borrow.

The crucial feature is that **interest does not earn interest**. Each year, the interest is calculated on the *original* principal only, not on any accumulated interest. This is like a landlord who charges the same monthly rent regardless of how long you have been a tenant — no compounding of the rent itself.

### Real-World Examples

- **Treasury bills (T-bills):** Short-term US government securities (maturities of 4, 8, 13, 26, or 52 weeks) are quoted on a discount basis, which is a form of simple interest.
- **Certificate of deposit penalties:** Some banks compute early withdrawal penalties using simple interest.
- **Short-term commercial loans:** For loans under one year, simple interest is common.
- **Credit card minimum payments:** The interest on a credit card balance for a single month is essentially simple interest on that month's balance.

> **Key Concept:** Simple interest grows **linearly** with time. If you earn $500 in year 1, you earn exactly $500 in year 2, year 3, and so on. The graph of balance versus time is a straight line, not a curve.

### The Formula

$$A = P(1 + rt)$$

where:
- $A$ = future value (what you will have at time $t$)
- $P$ = principal (your starting amount)
- $r$ = annual interest rate (as a decimal, e.g., 0.05 for 5%)
- $t$ = time in years

The interest earned is simply:

$$I = Prt$$

### Worked Example

You deposit $10,000 in a T-bill account paying 5% simple interest. How much do you have after 3 years?

$$A = 10{,}000 \times (1 + 0.05 \times 3) = 10{,}000 \times 1.15 = \$11{,}500$$

Interest earned: $I = 10{,}000 \times 0.05 \times 3 = \$1{,}500$

After 6 months (half a year):

$$A = 10{,}000 \times (1 + 0.05 \times 0.5) = 10{,}000 \times 1.025 = \$10{,}250$$

> **CFA Exam Tip:** On the CFA exam, time must be expressed in years. If you are given "6 months," use $t = 0.5$. If you are given "90 days," use $t = 90/365$ (or $t = 90/360$ depending on the day-count convention — more on that later).

> **Common Mistake:** Students sometimes confuse the interest *rate* with the interest *amount*. The rate $r = 0.05$ is a proportion. The interest *amount* is $I = Prt = \$1{,}500$. Make sure your final answer makes sense — if you compute interest of $150,000 on a $10,000 deposit, something went wrong.### When Is Simple Interest Used?

| Market | Convention | Typical instruments |
|:-------|:----------|:-------------------|
| **Money market** (< 1 year) | Simple interest | T-bills, commercial paper, repos |
| **Bonds** (> 1 year) | Compound interest | Government bonds, corporate bonds |
| **Consumer loans** | Usually compound | Mortgages, car loans, credit cards |

> **CFA Exam Tip:** Money market instruments (T-bills, CDs, commercial paper) use simple interest conventions. The "bank discount yield," "money market yield," and "bond equivalent yield" are all simple interest measures — know the difference between them.

> **Common Mistake:** Simple interest is NOT the same as "no interest." With simple interest, you still earn a return — the key difference is that you don't earn interest on accumulated interest. Over short periods (< 1 year), the difference between simple and compound is small. Over long periods, it becomes enormous.


The code below implements simple interest and demonstrates how the future value grows linearly with time. Notice how each additional year adds exactly the same dollar amount — this is the hallmark of simple (non-compounding) interest.
> **What to watch for:** The future value should grow as a straight line on a normal plot (linear growth). Compare this with the exponential curves you will see in the compound interest section — the visual difference is striking.


In [ ]:
def simple_interest(P, r, t):
    """Compute future value under simple interest.
    
    Parameters
    ----------
    P : float  -- principal (initial amount)
    r : float  -- annual interest rate (decimal, e.g. 0.05 for 5%)
    t : float  -- time in years
    
    Returns
    -------
    float -- future value A = P(1 + rt)
    """
    return P * (1.0 + r * t)


def simple_interest_earned(P, r, t):
    """Interest earned under simple interest: I = Prt."""
    return P * r * t


# --- Example: $10,000 at 5% for various terms ---
P = 10_000.0   # $10,000 principal
r = 0.05        # 5% annual rate
terms = np.array([0.25, 0.5, 1.0, 2.0, 5.0])  # years

print("Simple Interest: P = ${:,.0f}, r = {:.1%}".format(P, r))
print("-" * 50)
for t in terms:
    A = simple_interest(P, r, t)
    I = simple_interest_earned(P, r, t)
    print(f"  t = {t:5.2f} yr  =>  A = ${A:>10,.2f}   Interest = ${I:>8,.2f}")

**Interpreting the output:** Notice how the interest earned is perfectly proportional to time. At 0.25 years (3 months), you earn $125. At 1 year, exactly 4 times more: $500. At 5 years, exactly 20 times more: $2,500. This linear relationship is the defining property of simple interest — and the reason it underestimates the true growth of money over longer periods (where compound interest takes over).

### When Simple Interest Breaks Down

Simple interest is a reasonable approximation for short time periods, but it becomes increasingly inaccurate for longer horizons. Why? Because in reality, interest earned can be reinvested, and that reinvested interest itself earns interest. This "interest on interest" effect is exactly what compound interest captures.

For a 3-month T-bill, the difference between simple and compound interest is negligible. For a 30-year mortgage, ignoring compounding would produce wildly wrong results.> **Key Concept:** The linearity of simple interest means that the interest earned in year 10 is exactly the same as in year 1. With compound interest, the interest earned grows each year (because you earn interest on prior interest). This is why simple interest is only used for short periods — over long horizons, the lack of compounding significantly understates the true return.

### Simple vs Compound: The Growing Gap

For \$10,000 at 8%:
- After 1 year: Simple = \$10,800, Compound = \$10,800 (identical!)
- After 5 years: Simple = \$14,000, Compound = \$14,693 (\$693 gap)
- After 20 years: Simple = \$26,000, Compound = \$46,610 (\$20,610 gap!)

The gap grows exponentially because compound interest is exponential while simple interest is linear.


## 4. Compound Interest

### The Big Idea: Interest on Interest

Compound interest is one of the most powerful concepts in finance. Albert Einstein (apocryphally) called it the "eighth wonder of the world." The key difference from simple interest is that **earned interest is added to the principal**, and future interest is calculated on this larger amount.

Think of it as a snowball rolling downhill: it picks up snow (interest), gets bigger, and then picks up even *more* snow because it is larger. Over short periods, the snowball effect is barely noticeable. Over decades, it completely dominates.

### Real-World Analogy

Imagine you plant an apple tree. In Year 1, it produces 10 apples. With simple interest, you would get 10 apples every year forever. With compound interest, you plant those 10 apples as new trees, which themselves produce apples, which you plant again. After a few decades, you have an orchard — not because your original tree got more productive, but because the apples kept multiplying.

### The Formula

$$A = P\left(1 + \frac{r}{m}\right)^{mt}$$

where:
- $A$ = future value
- $P$ = principal
- $r$ = nominal (stated) annual rate
- $m$ = compounding frequency per year (e.g., 1 = annually, 2 = semi-annually, 12 = monthly, 365 = daily)
- $t$ = time in years

### Why Does Compounding Frequency Matter?

Consider 8% per year. The bank could calculate and add interest in different ways:

| Compounding | What Happens | Growth Factor (1 year) |
|-------------|-------------|----------|
| Annual ($m=1$) | Interest added once per year | $(1 + 0.08)^1 = 1.0800$ |
| Semi-annual ($m=2$) | 4% added every 6 months | $(1 + 0.04)^2 = 1.0816$ |
| Quarterly ($m=4$) | 2% added every 3 months | $(1 + 0.02)^4 = 1.0824$ |
| Monthly ($m=12$) | 0.667% added every month | $(1 + 0.00667)^{12} = 1.0830$ |
| Daily ($m=365$) | 0.0219% added every day | $(1 + 0.000219)^{365} = 1.0833$ |

The more frequently interest compounds, the more "interest on interest" you earn within the year, and the higher the effective annual return.

### Worked Example

You invest $10,000 at 8% compounded quarterly for 10 years.

**Step 1:** Identify the variables: $P = 10{,}000$, $r = 0.08$, $m = 4$, $t = 10$.

**Step 2:** Compute the periodic rate: $r/m = 0.08/4 = 0.02$ (2% per quarter).

**Step 3:** Compute the number of periods: $m \times t = 4 \times 10 = 40$ quarters.

**Step 4:** Apply the formula:
$$A = 10{,}000 \times (1.02)^{40} = 10{,}000 \times 2.20804 = \$22{,}080.40$$

Compare this to simple interest: $A_{\text{simple}} = 10{,}000 \times (1 + 0.08 \times 10) = \$18{,}000$.

The compounding "bonus" is $22{,}080 - 18{,}000 = \$4{,}080$ — that is the interest earned on interest.

### The Effective Annual Rate (EAR)

To fairly compare investments with different compounding frequencies, we convert everything to the **Effective Annual Rate (EAR)** — the actual annual rate of return after compounding.

$$\text{EAR} = \left(1 + \frac{r}{m}\right)^m - 1$$

For 8% compounded quarterly: $\text{EAR} = (1.02)^4 - 1 = 0.08243 = 8.243\%$. So 8% compounded quarterly is equivalent to 8.243% compounded annually.

> **Key Concept:** The **nominal rate** is the stated rate (8%). The **effective rate** is what you actually earn (8.243%). They are only equal when compounding is annual ($m = 1$). For all other compounding frequencies, EAR > nominal rate.

> **CFA Exam Tip:** The CFA curriculum calls the nominal rate the "stated annual rate" and the effective rate the "EAR" or "EAY" (effective annual yield). When a problem says "8% compounded quarterly," the 8% is the nominal rate, not the effective rate. Always check which one the question is asking for.### The Power of Compounding: A Historical Perspective

Legend has it that Albert Einstein called compound interest "the eighth wonder of the world." Whether or not he actually said this, the sentiment is correct. Consider:

| Years | Simple (8%) | Compound (8%) | Difference |
|:---:|:---:|:---:|:---:|
| 1 | $10,800 | $10,800 | $0 |
| 10 | $18,000 | $21,589 | $3,589 |
| 30 | $34,000 | $100,627 | $66,627 |
| 50 | $50,000 | $469,016 | $419,016 |

After 50 years, compound interest produces nearly **10x** what simple interest produces!

> **Key Concept:** The "miracle" of compounding is that your interest earns interest, which earns more interest, creating exponential growth. The formula $A = P(1 + r/m)^{mt}$ produces exponential growth in $t$, while simple interest $A = P(1 + rt)$ produces only linear growth.

### Effective Annual Rate (EAR)

Different compounding frequencies make direct comparison difficult. The **Effective Annual Rate** solves this by expressing any rate as its equivalent annual compound rate:

$$\text{EAR} = \left(1 + \frac{r_{\text{nom}}}{m}\right)^m - 1$$

**Worked Example:** A bank offers 8% compounded quarterly. What is the EAR?

$$\text{EAR} = \left(1 + \frac{0.08}{4}\right)^4 - 1 = (1.02)^4 - 1 = 0.08243 = 8.243\%$$

The effective rate is higher than the nominal rate because of intra-year compounding.

> **CFA Exam Tip:** The EAR is the correct rate for comparing investments with different compounding frequencies. The exam often gives nominal rates with different compounding and asks which investment has the higher effective return. Always convert to EAR first.
### The Rule of 72

A useful mental shortcut for compound interest:

$$\text{Years to double} \approx \frac{72}{r\%}$$

| Rate | Rule of 72 | Exact | Error |
|:---:|:---:|:---:|:---:|
| 4% | 18.0 years | 17.7 years | 1.7% |
| 6% | 12.0 years | 11.9 years | 0.8% |
| 8% | 9.0 years | 9.0 years | 0.0% |
| 12% | 6.0 years | 6.1 years | 1.7% |

> **CFA Exam Tip:** The Rule of 72 is a quick way to estimate doubling time in your head. It's most accurate around 8% and slightly overstates the time at very low or very high rates. It also works in reverse: at 6%, your purchasing power halves in 72/6 = 12 years due to inflation.


Let us verify these calculations with code and visualize how different compounding frequencies affect growth over time.

In [ ]:
def compound_interest(P, r, t, m=1):
    """Future value under compound interest.
    
    Parameters
    ----------
    P : float  — principal
    r : float  — nominal annual rate
    t : float  — time in years
    m : int    — compounding frequency per year
    """
    return P * (1.0 + r / m) ** (m * t)


def effective_annual_rate(r_nom, m):
    """Convert nominal rate to effective annual rate.
    
    EAR = (1 + r_nom/m)^m - 1
    This is the 'true' annual return after accounting for compounding.
    """
    return (1.0 + r_nom / m) ** m - 1.0


# --- Compare compounding frequencies ---
P = 10_000.0
r = 0.08  # 8% nominal
t = 10.0  # 10 years

frequencies = {'Annual (m=1)': 1, 'Semi-annual (m=2)': 2,
               'Quarterly (m=4)': 4, 'Monthly (m=12)': 12,
               'Daily (m=365)': 365}

print(f"Compound Interest: P = ${P:,.0f}, r = {r:.1%}, t = {t:.0f} years")
print("-" * 65)
for label, m in frequencies.items():
    A = compound_interest(P, r, t, m)
    ear = effective_annual_rate(r, m)
    print(f"  {label:<25s}  FV = ${A:>12,.2f}   EAR = {ear:.6%}")

**Interpreting the output:** All five scenarios start with the same $10,000 and the same 8% nominal rate. Yet after 10 years, daily compounding yields about $664 more than annual compounding. The EAR column makes the comparison clear: 8% compounded daily is effectively 8.3278%, while 8% compounded annually is just 8%.

The visualization below shows two important insights:
1. **Left panel:** Growth curves for different compounding frequencies. The gaps between curves widen over time — compounding's advantage gets bigger the longer you wait.
2. **Right panel:** Diminishing returns of more frequent compounding. Going from annual to monthly makes a big difference; going from monthly to daily barely matters.> **Key Concept:** The gap between annual and continuous compounding is the EAR difference: $e^{0.08} - 1.08 = 0.0033$, or about 33 basis points. For a \$10,000 investment over 10 years, this translates to roughly \$70. Small for individuals, but for a bank managing \$10 billion, the difference is \$3.3 million per year.

> **CFA Exam Tip:** When comparing investments with different compounding frequencies, ALWAYS convert to the same basis (EAR is the standard) before comparing. A "12% compounded monthly" investment is better than a "12.1% compounded annually" investment because the EAR of the first is $(1.01)^{12} - 1 = 12.68\%$.


In [ ]:
# --- Visualization: Growth curves for different compounding frequencies ---
t_grid = np.linspace(0, 20, 500)
P, r = 1.0, 0.08

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: growth curves
ax = axes[0]
colors = [PRIMARY, SECONDARY, TERTIARY, ACCENT, 'purple']
for (label, m), c in zip(frequencies.items(), colors):
    A_curve = compound_interest(P, r, t_grid, m)
    ax.plot(t_grid, A_curve, label=label, color=c, linewidth=2)

# Continuous limit
A_cont = P * np.exp(r * t_grid)
ax.plot(t_grid, A_cont, '--', label='Continuous', color='black', linewidth=2)
ax.set_xlabel('Time (years)')
ax.set_ylabel('Growth factor A/P')
ax.set_title('Growth Under Different Compounding Frequencies')
ax.legend(fontsize=9)

# Right panel: EAR vs compounding frequency
ax = axes[1]
m_values = np.arange(1, 366)
ear_values = effective_annual_rate(r, m_values)
ax.plot(m_values, ear_values * 100, color=PRIMARY, linewidth=2)
ax.axhline(y=(np.exp(r) - 1) * 100, color=SECONDARY, linestyle='--',
           label=f'Continuous limit: {(np.exp(r)-1)*100:.4f}%')
ax.set_xlabel('Compounding frequency m')
ax.set_ylabel('Effective Annual Rate (%)')
ax.set_title(f'EAR vs Compounding Frequency (r = {r:.0%})')
ax.legend()
ax.set_xscale('log')

plt.tight_layout()
plt.show()

**What the plots tell us:** The left panel shows that the difference between annual and continuous compounding is modest over short horizons but becomes very significant over decades. The right panel shows a key insight: the EAR rises rapidly as you go from annual to monthly compounding, but the incremental benefit of compounding more frequently than monthly is tiny. This is why the continuous compounding limit (the dashed line) is so useful — it is essentially the ceiling.

> **Key Concept:** Compounding frequency has **diminishing marginal returns**. The jump from annual to semi-annual is much larger than the jump from monthly to daily. Beyond monthly compounding, you are very close to the theoretical maximum (continuous compounding).

> **CFA Exam Tip:** If a question asks you to rank investments by effective return, convert all to EAR first. The investment with the highest EAR is the best deal, regardless of how the nominal rate is quoted.The right panel (log scale) reveals the exponential nature of compound growth — on a log scale, exponential growth appears as a straight line. The slope of the line is the continuously compounded rate. This is one reason why finance professionals prefer log-returns: they make exponential growth linear and additive.

> **Common Mistake:** Don't confuse the nominal rate with the growth rate. With annual compounding, \$10,000 at 8% grows to $10,000 \times 1.08^{10} = \$21,589$ after 10 years. The \$11,589 of interest earned is NOT 80% of the principal (which would be 8% × 10 years under simple interest). It's 115.9% — the extra 35.9% is the "interest on interest" from compounding.
> **Key Concept:** The convergence to $e^{rT}$ is rapid. Monthly compounding ($m = 12$) already captures most of the benefit — the remaining gap to continuous is tiny. This is why in practice, the difference between monthly and continuous compounding is negligible for most purposes.

The mathematical reason: the Taylor expansion of $(1 + r/m)^{mT}$ converges to $e^{rT}$ at rate $O(1/m)$. By $m = 12$, you're already within a few basis points of the limit.


## 5. Continuous Compounding

### The Idea: Compound Every Instant

What happens if we compound not annually, not monthly, not daily, but *continuously* — at every instant in time? This might sound like a theoretical curiosity, but continuous compounding is actually the most convenient convention for many financial calculations. It simplifies formulas, makes calculus straightforward, and is the standard in derivatives pricing (the Black-Scholes model uses continuous rates).

### The Mathematical Derivation

Start with the compound interest formula and let $m \to \infty$:

$$\lim_{m \to \infty} P\left(1 + \frac{r}{m}\right)^{mt}$$

Substitute $n = m/r$:

$$= P \left[\left(1 + \frac{1}{n}\right)^n\right]^{rt}$$

The term in brackets converges to $e = 2.71828\ldots$ as $n \to \infty$:

$$\lim_{n \to \infty}\left(1 + \frac{1}{n}\right)^n = e$$

Therefore:

$$A = Pe^{rt}$$

This is the **continuous compounding formula**. It says that money grows exponentially at rate $r$.

### Why Practitioners Use Continuous Compounding

1. **Mathematical convenience:** The exponential function $e^{rt}$ is its own derivative, making calculus-based finance much cleaner.

2. **Additivity of rates:** Under continuous compounding, the return over two consecutive periods is simply the sum of the two rates. This property makes continuous rates the natural choice for stochastic calculus models.

3. **Industry standard in derivatives:** The Black-Scholes formula, interest rate swap pricing, and most quantitative models use continuous rates.

### Worked Example

You invest $1,000 at a continuously compounded rate of 10% for 5 years.

$$A = 1{,}000 \times e^{0.10 \times 5} = 1{,}000 \times e^{0.5} = 1{,}000 \times 1.6487 = \$1{,}648.72$$

The present value formula works in reverse:

$$P = A \times e^{-rt} = 1{,}648.72 \times e^{-0.5} = \$1{,}000$$

> **Key Concept:** The continuously compounded rate is always *lower* than the equivalent nominal rate (for the same effective return). For example, 10% continuous is equivalent to about 10.52% annually compounded. The continuous rate is "doing more compounding work" with a smaller number.

> **Common Mistake:** Do not use $e^{rt}$ when the problem states "compounded semi-annually" or "compounded annually." Continuous compounding is a specific convention. Only use the exponential formula when the rate is explicitly stated as continuously compounded.### Why Finance Uses Continuous Compounding

Continuous compounding is the mathematical ideal — it's elegant and simplifies many formulas:

| Property | Discrete compounding | Continuous compounding |
|:---------|:---:|:---:|
| Future value | $P(1 + r/m)^{mt}$ | $Pe^{rt}$ |
| Present value | $P/(1 + r/m)^{mt}$ | $Pe^{-rt}$ |
| Rate addition | Complex | Simply add: $e^{r_1 t} \cdot e^{r_2 t} = e^{(r_1+r_2)t}$ |
| Multi-period | Complex | Simply multiply exponents |

> **Key Concept:** Continuous compounding is used in theoretical finance (Black-Scholes, GBM, forward rates) because the exponential function has beautiful mathematical properties. Discrete compounding is used in practice (bank accounts, mortgages, bond coupons). You need to convert between them fluently.

**Conversion formulas:**
- Discrete → Continuous: $r_c = m \ln(1 + r_{\text{nom}}/m)$
- Continuous → Discrete: $r_{\text{nom}} = m(e^{r_c/m} - 1)$

> **Common Mistake:** When the Black-Scholes formula uses $r$, it means the **continuously compounded** risk-free rate. If you're given a rate compounded semi-annually (like most Treasury bonds), you must convert it first: $r_c = 2\ln(1 + r_{\text{sa}}/2)$.
### Where Does $e$ Come From?

The number $e \approx 2.71828$ emerges naturally from compounding:

$$e = \lim_{m \to \infty} \left(1 + \frac{1}{m}\right)^m$$

This is why $e^{rT}$ appears in the continuous compounding formula — it's the limiting case of $(1 + r/m)^{mT}$ as $m \to \infty$. The number $e$ is not an arbitrary choice; it's the unique base for which the derivative of $b^x$ is $b^x$ itself, making calculus with exponentials and logarithms clean and elegant.


The following code verifies that as the compounding frequency $m$ increases, the compound interest formula converges to $Pe^{rt}$.

In [ ]:
def continuous_fv(P, r, t):
    """Future value under continuous compounding: A = P * exp(r*t)."""
    return P * np.exp(r * t)


def continuous_pv(A, r, t):
    """Present value under continuous compounding: P = A * exp(-r*t)."""
    return A * np.exp(-r * t)


# --- Verify the limit: discrete compounding converges to continuous ---
P, r, t = 1000.0, 0.10, 5.0
exact = continuous_fv(P, r, t)

print(f"Continuous FV (exact): ${exact:,.6f}")
print()
for m in [1, 10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    approx = compound_interest(P, r, t, m)
    err = abs(approx - exact)
    print(f"  m = {m:>10,d}   FV = ${approx:>14,.6f}   |error| = ${err:.2e}")

**Interpreting the output:** With annual compounding ($m = 1$), the error is about $17. By monthly compounding ($m = 12$), we are within a few cents. By $m = 1{,}000$, the error is less than a penny. The convergence is clear and confirms the mathematical limit.

### A Handy Mental Model

Think of the relationship between discrete and continuous compounding like this:

- **Discrete** compounding is like a staircase — the balance jumps up at each compounding date.
- **Continuous** compounding is like a smooth ramp — the balance increases constantly, every nanosecond.

As you add more stairs (higher $m$), the staircase looks more and more like a ramp. In the limit, it *is* the ramp.

## 6. Rate Conversions

### Why Rate Conversions Matter

In practice, different financial instruments quote rates using different conventions:

- A **savings account** might quote 5% compounded monthly
- A **bond** might quote 6% semi-annually
- A **derivatives desk** might quote 4.5% continuously compounded
- A **credit card** might advertise 18% APR (= nominal rate, compounded monthly)

To compare these instruments fairly, you must be able to convert between conventions. It is like comparing temperatures in Fahrenheit and Celsius — the underlying reality (how hot it is) does not change, but the *scale* does.

### The Three Rate Types

| Rate Type | Symbol | Description | When Used |
|-----------|--------|-------------|----------|
| Nominal (stated) rate | $r_{\text{nom}}$ | Annual rate divided equally across $m$ periods | Bond coupons, loan rates |
| Effective annual rate | $r_{\text{eff}}$ | True annual return after all compounding | Comparing investments |
| Continuous rate | $r_c$ | Limit as $m \to \infty$ | Derivatives pricing, quant models |

### Conversion Formulas

**Nominal to Effective:**
$$r_{\text{eff}} = \left(1 + \frac{r_{\text{nom}}}{m}\right)^m - 1$$

**Effective to Nominal:**
$$r_{\text{nom}} = m\left[(1 + r_{\text{eff}})^{1/m} - 1\right]$$

**Nominal to Continuous:**
$$r_c = m \ln\left(1 + \frac{r_{\text{nom}}}{m}\right)$$

**Continuous to Effective:**
$$r_{\text{eff}} = e^{r_c} - 1$$

### Worked Example: Which Account Pays More?

Bank A offers 5.9% compounded monthly. Bank B offers 6.0% compounded semi-annually. Which is better?

**Bank A:** $\text{EAR}_A = (1 + 0.059/12)^{12} - 1 = (1.004917)^{12} - 1 = 0.06062 = 6.062\%$

**Bank B:** $\text{EAR}_B = (1 + 0.06/2)^{2} - 1 = (1.03)^{2} - 1 = 0.0609 = 6.09\%$

Bank B wins despite having a seemingly similar nominal rate, because the higher per-period rate more than compensates for less frequent compounding.

> **CFA Exam Tip:** When comparing investments, ALWAYS convert to the same basis first. The EAR is the universal comparison tool. A common exam question will ask you to rank investments with different compounding frequencies — the one with the highest EAR is the best deal.

> **Common Mistake:** Treating the nominal rate as if it were the effective rate. An 8% nominal rate compounded monthly is NOT an 8% annual return — it is an 8.30% annual return.### The Conversion Roadmap

The three rate types form a triangle — you need to convert freely between all three:

```
         Nominal (stated)
          /           \
  EAR formula    r_c = m*ln(1+r_nom/m)
        /               \
  Effective  ←————→  Continuous
        EAR = e^(r_c) - 1
```

**Worked Example:** A bond yields 6% compounded semi-annually. Express as (a) EAR, (b) continuous.

(a) $\text{EAR} = (1 + 0.03)^2 - 1 = 6.09\%$

(b) $r_c = 2\ln(1.03) = 2 \times 0.02956 = 5.912\%$

**Verification:** $e^{0.05912} = 1.0609$ ✓ (matches the EAR)

> **CFA Exam Tip:** Know the hierarchy: **continuous < nominal < EAR** (for the same investment). Continuous is always the lowest quoted rate; EAR is always the highest. This is because more frequent compounding produces more "interest on interest," so you need a lower stated rate to produce the same effective growth.


The code below implements all six conversion functions and demonstrates round-trip consistency — converting from one convention to another and back should return the original rate.

In [ ]:
def nominal_to_effective(r_nom, m):
    """Nominal rate (compounded m times/yr) -> effective annual rate."""
    return (1.0 + r_nom / m) ** m - 1.0

def effective_to_nominal(r_eff, m):
    """Effective annual rate -> nominal rate compounded m times/yr."""
    return m * ((1.0 + r_eff) ** (1.0 / m) - 1.0)

def nominal_to_continuous(r_nom, m):
    """Nominal rate -> continuously compounded rate."""
    return m * np.log(1.0 + r_nom / m)

def continuous_to_nominal(r_c, m):
    """Continuously compounded rate -> nominal rate compounded m times/yr."""
    return m * (np.exp(r_c / m) - 1.0)

def effective_to_continuous(r_eff):
    """Effective annual rate -> continuously compounded rate."""
    return np.log(1.0 + r_eff)

def continuous_to_effective(r_c):
    """Continuously compounded rate -> effective annual rate."""
    return np.exp(r_c) - 1.0


# --- Conversion example and round-trip verification ---
r_nom = 0.08  # 8% nominal semi-annual
m = 2

r_eff = nominal_to_effective(r_nom, m)
r_c = nominal_to_continuous(r_nom, m)

print(f"Starting: Nominal rate r = {r_nom:.4%} compounded {m}x/year")
print(f"  Effective annual rate:         {r_eff:.6%}")
print(f"  Continuously compounded rate:  {r_c:.6%}")
print()

# Round-trip verification
r_nom_back = effective_to_nominal(r_eff, m)
r_c_back = effective_to_continuous(r_eff)
print("Round-trip verification:")
print(f"  EAR -> Nominal(m=2):      {r_nom_back:.6%}  (should be {r_nom:.6%})")
print(f"  EAR -> Continuous:         {r_c_back:.6%}  (should be {r_c:.6%})")
print()

# Comprehensive table
print("\nConversion Table: 8% nominal at various compounding frequencies")
print("-" * 70)
print(f"{'Frequency':<20s} {'Nominal':>10s} {'Effective':>12s} {'Continuous':>12s}")
print("-" * 70)
for label, m in [('Annual', 1), ('Semi-annual', 2), ('Quarterly', 4),
                  ('Monthly', 12), ('Daily', 365)]:
    r_e = nominal_to_effective(r_nom, m)
    r_cc = nominal_to_continuous(r_nom, m)
    print(f"{label:<20s} {r_nom:>10.4%} {r_e:>12.6%} {r_cc:>12.6%}")

**Interpreting the output:** Starting from the same 8% nominal rate, the effective annual rate increases with compounding frequency (from exactly 8% for annual to 8.3278% for daily), while the continuously compounded equivalent *decreases* (from 8% for annual down to 7.9726% for daily). This makes sense: a lower continuous rate can produce the same annual growth because it compounds more frequently.

The round-trip verification confirms that our conversion functions are consistent — converting to EAR and back recovers the original nominal rate exactly (up to floating-point precision).

> **Key Concept:** All three rate types — nominal, effective, and continuous — describe the *same* economic reality (the growth of money over one year). They are just different rulers measuring the same length. The conversion formulas let you translate between rulers.> **Important:** The relationship between the three rates for the same investment is always:

$$r_{\text{continuous}} < r_{\text{nominal}} < \text{EAR}$$

(assuming compounding more than once per year). This ordering is a useful sanity check when doing conversions.

### Round-Trip Verification

A critical test for any rate conversion implementation: converting from nominal → EAR → continuous → back to nominal should return the original rate. If it doesn't, there's a bug. Let's verify this in the code above.


## 7. Discount Factors

### The Idea: How Much Is Future Money Worth Today?

If the future value formula answers "How much will my money grow to?", the **discount factor** answers the reverse question: "How much is a future payment worth *right now*?"

This is the single most important concept in valuation. Every asset in finance — bonds, stocks, real estate, companies — is valued by estimating its future cash flows and **discounting** them back to the present.

### Real-World Analogy

Suppose someone offers to pay you $1,000 in exactly 5 years. How much would you pay for that promise today? If you could earn 6% annually on your money, then $1,000 in 5 years is worth only about $747 today — because $747 invested at 6% would grow to exactly $1,000 in 5 years.

The **discount factor** $d(t)$ tells you: for every dollar promised at time $t$, how many dollars is it worth today?

### The Formulas

**Under continuous compounding:**
$$d(t) = e^{-rt}$$

**Under annual compounding:**
$$d(t) = (1 + r)^{-t}$$

where:
- $d(t)$ = discount factor for time $t$ (always between 0 and 1 for positive rates)
- $r$ = interest rate (appropriate for the compounding convention)
- $t$ = time in years

### Key Properties of Discount Factors

1. $d(0) = 1$ — money today is worth 100% of its face value.
2. $d(t) < 1$ for $t > 0$ (assuming positive rates) — future money is worth less than present money.
3. $d(t)$ decreases as $t$ increases — money further in the future is worth less.
4. $d(t)$ decreases as $r$ increases — higher rates mean future money is worth even less today.

### Worked Example

What is the present value of $1,000 due in 3 years if the continuously compounded rate is 5%?

$$d(3) = e^{-0.05 \times 3} = e^{-0.15} = 0.8607$$

$$PV = 1{,}000 \times 0.8607 = \$860.71$$

### Discount Factors and Bond Pricing

The price of any bond is simply:

$$P = \sum_{i=1}^{n} CF_i \times d(t_i)$$

Each cash flow is multiplied by the discount factor for its payment date.

> **Key Concept:** A **discount factor curve** is mathematically equivalent to a **yield curve**. They carry exactly the same information, just expressed differently. Knowing one, you can derive the other.

> **CFA Exam Tip:** If given a spot rate curve and asked to price a bond, first compute the discount factor for each cash flow date, then multiply each cash flow by its discount factor and sum. This is the most reliable and error-free approach.### The Discount Factor as the Price of a Zero-Coupon Bond

> **Key Concept:** The discount factor $d(T)$ is literally the **price today of receiving $1 at time $T$**. If $d(5) = 0.78$, you would pay \$0.78 today for a promise of \$1.00 in 5 years.

This means a zero-coupon bond with face value $F$ maturing at time $T$ has price:
$$P = F \times d(T)$$

And ANY bond (with coupons) can be priced by discounting each cash flow:
$$P = \sum_{i=1}^n C_i \times d(t_i)$$

This is the **fundamental pricing equation** of fixed income.

### Discount Factor Curve

The set of discount factors for all maturities $\{d(t) : t > 0\}$ is called the **discount function** or **discount curve**. It always:
- Starts at $d(0) = 1$ (a dollar today is worth a dollar)
- Decreases with maturity (a dollar far in the future is worth less)
- Is always positive (money always has some value)

> **Common Mistake:** Don't confuse the discount factor with the discount rate. The discount factor is a *price* (between 0 and 1). The discount rate is an *interest rate* (a percentage). They are inversely related: higher rates → lower discount factors.


The following code computes and visualizes discount factor curves for different interest rate levels, and shows the relationship between spot rates and discount factors.> **What to watch for:** The discount factor curves should all start at 1.0 (at $t = 0$) and decay toward zero. Higher interest rates cause faster decay. The continuous, semi-annual, and annual curves for the same rate should be nearly identical — the compounding convention makes only a small difference.


In [ ]:
def discount_factor(r, t, compounding='continuous'):
    """Compute discount factor.
    
    Parameters
    ----------
    r : float or array — interest rate
    t : float or array — time in years
    compounding : str — 'continuous' or 'annual'
    """
    if compounding == 'continuous':
        return np.exp(-r * t)
    elif compounding == 'annual':
        return (1.0 + r) ** (-t)
    else:
        raise ValueError(f"Unknown compounding: {compounding}")


def zero_rate_from_df(df, t):
    """Extract continuously compounded zero rate from discount factor."""
    return -np.log(df) / t


# --- Discount factor curves for different rates ---
t_grid = np.linspace(0.01, 30, 300)
rates = [0.02, 0.04, 0.06, 0.08, 0.10]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for r_val in rates:
    df_curve = discount_factor(r_val, t_grid)
    ax.plot(t_grid, df_curve, linewidth=2, label=f'r = {r_val:.0%}')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Discount Factor d(t)')
ax.set_title('Discount Factor Curves')
ax.legend()

# Build a non-flat discount curve and extract zero rates
ax = axes[1]
# Simulate a term structure: rates increase with maturity
zero_rates_curve = 0.02 + 0.03 * (1 - np.exp(-0.5 * t_grid))
df_curve = discount_factor(zero_rates_curve, t_grid)

ax.plot(t_grid, zero_rates_curve * 100, color=PRIMARY, linewidth=2, label='Zero rates')
ax2 = ax.twinx()
ax2.plot(t_grid, df_curve, color=SECONDARY, linewidth=2, label='Discount factors')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Zero Rate (%)', color=PRIMARY)
ax2.set_ylabel('Discount Factor', color=SECONDARY)
ax.set_title('Zero Rate Curve and Discount Factors')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

**Interpreting the output:**

- **Left panel:** Higher interest rates cause discount factors to decay faster. At 10%, $1 due in 30 years is worth only about 5 cents today. At 2%, it is worth about 55 cents. This is why long-duration assets (like 30-year bonds) are so sensitive to interest rate changes.
- **Right panel:** The zero rate curve and discount factor curve are mirror images of each other — as rates rise with maturity (upward-sloping curve), discount factors decline more steeply.

### Practical Implications

For a portfolio manager, the discount factor curve is the central tool for valuation:

- **Rising rates** mean lower discount factors, which mean lower present values, which means **bond prices fall**.
- **Long-dated cash flows** are more affected by rate changes because the discount factor is more sensitive at longer maturities.
- This is exactly why "duration" measures interest rate sensitivity — it captures how much the weighted-average discount factor changes.> **CFA Exam Tip:** The discount factor is the reciprocal of the accumulation factor:
> $$d(T) = \frac{1}{(1+r)^T} = (1+r)^{-T}$$
> For continuous compounding: $d(T) = e^{-rT}$. The discount factor answers: "What is \$1 received at time $T$ worth today?" It is always between 0 and 1 (assuming positive rates).

> **Common Mistake:** When rates are negative (as occurred in Europe 2014–2022), the discount factor exceeds 1 — meaning a future dollar is worth MORE than a dollar today. This seems paradoxical but is simply the mathematical consequence of negative rates: you are penalised for lending (receiving less back than you put in).


## 8. Forward Rates

### The Idea: Locking In Future Borrowing Costs

Imagine you know you will need to borrow money in 2 years for a 1-year project. You can see today's interest rates for 2-year and 3-year loans. Can you figure out what rate you could lock in *today* for that future 1-year loan? Yes — and that rate is called the **forward rate**.

### Real-World Analogy

Think of airline tickets. The spot rate is like buying a ticket for a flight today. The forward rate is like buying a ticket *now* for a flight that departs in 6 months. The price is determined by the market's expectation of what a walk-up ticket will cost, plus or minus a premium for the certainty of locking in the price.

### The No-Arbitrage Argument

Forward rates are **implied by current spot rates** through a no-arbitrage argument:

**Strategy A:** Invest for 3 years at the 3-year spot rate $r_3$.

**Strategy B:** Invest for 2 years at $r_2$, then roll over for 1 year at whatever rate prevails.

The forward rate $f(2,3)$ makes these equivalent:

$$e^{r_3 \times 3} = e^{r_2 \times 2} \times e^{f(2,3) \times 1}$$

Solving (continuous compounding):

$$f(t_1, t_2) = \frac{r_2 \cdot t_2 - r_1 \cdot t_1}{t_2 - t_1}$$

### Worked Example

The 2-year spot rate is 3.5% and the 3-year spot rate is 4.0%. What is the 1-year forward rate starting in 2 years?

$$f(2,3) = \frac{0.04 \times 3 - 0.035 \times 2}{3 - 2} = \frac{0.12 - 0.07}{1} = 0.05 = 5.0\%$$

**Interpretation:** The market is pricing in a 5% rate for the third year. If the average rate over 3 years is 4%, and the first 2 years average 3.5%, the third year must pull the average up.

### Spot Rates vs Forward Rates: A GPA Analogy

The spot rate $r(t)$ is like a **cumulative GPA** through semester $t$. The forward rate $f(t)$ is like a **single semester GPA**. If your cumulative GPA is rising, your latest semester must be above the cumulative.

> **Key Concept:** Forward rates are **not forecasts** of where future spot rates will actually go. They are rates implied by today's term structure through no-arbitrage. They incorporate expectations, risk premia, and supply/demand effects.

> **CFA Exam Tip:** "If the yield curve is upward-sloping, what can we infer about forward rates?" Answer: Forward rates are above spot rates at every maturity. But this does NOT necessarily mean the market expects rates to rise — it could reflect a liquidity premium.### The No-Arbitrage Argument

Forward rates are determined by no-arbitrage. Here's the argument:

**Setup:** You want to invest from year 1 to year 2. Two strategies:

**Strategy A:** Wait until year 1, invest at whatever the spot rate is then → uncertain return

**Strategy B:** Today, lock in the forward rate $f(1,2)$:
1. Borrow for 1 year at $s_1$ (owe $(1+s_1)$ at year 1)
2. Invest for 2 years at $s_2$ (receive $(1+s_2)^2$ at year 2)
3. Net: invested $(1+s_1)$ at year 1, received $(1+s_2)^2$ at year 2

The locked-in rate:
$$f(1,2) = \frac{(1+s_2)^2}{(1+s_1)} - 1$$

> **Key Concept:** The forward rate is NOT a prediction of future rates. It's the rate you can GUARANTEE today via the arbitrage-free replication above. Whether actual rates turn out higher or lower is irrelevant — you've already locked in $f(1,2)$.

### Worked Example

$s_1 = 3\%$, $s_2 = 4\%$. What is $f(1,2)$?

$$f(1,2) = \frac{(1.04)^2}{1.03} - 1 = \frac{1.0816}{1.03} - 1 = 5.01\%$$

The forward rate (5.01%) exceeds both spot rates because it must "make up" for the lower 1-year rate over the first year.

> **CFA Exam Tip:** When the spot curve is upward-sloping, forward rates lie ABOVE the spot curve. When the spot curve is flat, forward rates EQUAL spot rates. When the spot curve is downward-sloping, forward rates lie BELOW the spot curve.


The code below computes forward rates from our spot rate curve and visualizes the relationship between spot and instantaneous forward rates.> **What to watch for:** When the spot curve is upward-sloping, the forward curve should lie ABOVE it. The gap between forward and spot rates reveals the steepness of the term structure — a large gap means the curve is steepening rapidly at that maturity.


In [ ]:
def forward_rate(r1, t1, r2, t2):
    """Compute forward rate f(t1, t2) from continuously compounded spot rates.
    
    Parameters
    ----------
    r1, r2 : float — spot rates for maturities t1, t2
    t1, t2 : float — maturities in years (t2 > t1)
    """
    return (r2 * t2 - r1 * t1) / (t2 - t1)


def forward_rate_from_df(df1, t1, df2, t2):
    """Compute forward rate from discount factors."""
    return -np.log(df2 / df1) / (t2 - t1)


def instantaneous_forward(t, r_func, dt=1e-6):
    """Numerical instantaneous forward rate: f(t) = d/dt [r(t)*t]."""
    rt_plus = r_func(t + dt) * (t + dt)
    rt_minus = r_func(t - dt) * (t - dt)
    return (rt_plus - rt_minus) / (2.0 * dt)


# --- Example: upward-sloping term structure ---
# Define a spot rate curve: r(t) = 0.02 + 0.03*(1 - exp(-0.5t))
def spot_rate(t):
    return 0.02 + 0.03 * (1.0 - np.exp(-0.5 * t))

# Compute forward rates for various intervals
print("Forward Rates from Spot Curve")
print("-" * 55)
print(f"{'Period':<15s} {'r(t1)':>8s} {'r(t2)':>8s} {'f(t1,t2)':>10s}")
print("-" * 55)
intervals = [(0.5, 1.0), (1.0, 2.0), (2.0, 3.0), (3.0, 5.0), (5.0, 10.0)]
for t1, t2 in intervals:
    r1, r2 = spot_rate(t1), spot_rate(t2)
    f = forward_rate(r1, t1, r2, t2)
    print(f"  [{t1:.1f}, {t2:.1f}]     {r1:>8.4%} {r2:>8.4%} {f:>10.4%}")

# Visualize spot vs forward
t_grid = np.linspace(0.1, 20, 500)
spot_curve = spot_rate(t_grid)
fwd_curve = np.array([instantaneous_forward(t, spot_rate) for t in t_grid])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(t_grid, spot_curve * 100, color=PRIMARY, linewidth=2, label='Spot rate r(t)')
ax.plot(t_grid, fwd_curve * 100, color=SECONDARY, linewidth=2, label='Instantaneous forward f(t)')
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Rate (%)')
ax.set_title('Spot Rate vs Instantaneous Forward Rate')
ax.legend()
plt.tight_layout()
plt.show()

**Interpreting the output:**

The **forward rate curve always lies above the spot rate curve** when the curve is upward-sloping. The spot rate is an *average* of all forward rates up to that maturity. If the average is increasing, the marginal rate must be above the average.

### What Forward Rates Tell Investors

- **Upward-sloping forward curve:** The market prices in higher short-term rates in the future (or investors demand a term premium).
- **Flat forward curve:** The market expects short-term rates to remain roughly constant.
- **Downward-sloping forward curve:** The market prices in lower rates — often a recession signal.

Central banks and bond traders closely monitor forward rate curves because they embed the market's collective view about future monetary policy.> **Key Concept:** Forward rates are the marginal rates — the rate for each incremental period. If the spot curve is upward-sloping, each successive year must have a higher "marginal" rate to pull up the average. Think of it like grade averages: if your GPA is rising each year, each new year's grades must be above your cumulative average.

### Forward Rates and Expectations

Under the **expectations hypothesis**, forward rates equal expected future spot rates. If $f(1,2) = 5\%$ and the expectations hypothesis holds, then the market expects the 1-year spot rate one year from now to be 5\%.

In practice, forward rates overpredict future spot rates due to the term premium. Still, changes in forward rates are informative about changes in rate expectations.

> **CFA Exam Tip:** The relationship between spot and forward rates is heavily tested:
> - $(1 + s_n)^n = (1 + s_1)(1 + f_{1,2})(1 + f_{2,3})\cdots(1 + f_{n-1,n})$
> - This says: investing at the $n$-year spot rate must equal rolling over 1-year forward rates


## 9. Day Count Conventions

### Why Do Day Count Conventions Exist?

Day count conventions seem like a tedious technicality, but they have real financial impact. When a bond pays a 6% coupon semi-annually, and you buy it between coupon dates, you must pay the seller for accrued interest. But exactly how many days of interest?

### Real-World Context: Which Markets Use Which Convention?

| Convention | Markets | How It Works |
|-----------|---------|-------------|
| **30/360** | US corporate bonds, US agency bonds, most mortgages | Pretends every month has 30 days and every year has 360 days |
| **ACT/360** | US money markets (LIBOR, Fed Funds), Euro commercial paper | Actual days elapsed, but divides by 360 |
| **ACT/365** | UK government bonds (gilts), Canadian bonds | Actual days elapsed, divides by 365 |
| **ACT/ACT** | US Treasury bonds, most European government bonds | Actual days elapsed, divides by actual days in the year |

### Why Do These Differences Matter?

Consider a $1 million position earning 5% interest from January 1 to July 1 (181 days in a non-leap year):

| Convention | Year Fraction | Accrued Interest |
|-----------|:------------:|:----------------:|
| 30/360 | 180/360 = 0.5000 | $25,000.00 |
| ACT/360 | 181/360 = 0.5028 | $25,138.89 |
| ACT/365 | 181/365 = 0.4959 | $24,794.52 |
| ACT/ACT | 181/365 = 0.4959 | $24,794.52 |

The difference between the highest and lowest is **$344** on $1 million for just 6 months. On a $100M portfolio, that is $34,400.

### Understanding Each Convention

**30/360 (Bond Basis):** Assumes each month has 30 days. Dates back to pre-computer era when uniform months simplified manual calculations.

**ACT/360 (Money Market Basis):** Uses actual elapsed days but divides by 360. A full year gives 365/360 = 1.01389, so lenders earn slightly more than the quoted rate.

**ACT/365 (Fixed):** Straightforward — count actual days, divide by 365, even in leap years.

**ACT/ACT (ISDA):** The most "honest" convention — actual days divided by actual days in the year.

### Worked Example

Accrued interest on a US corporate bond ($1M face, 6% coupon, semi-annual) purchased March 15, 2024. Last coupon was January 15, 2024.

**Step 1:** US corporate bonds use 30/360.

**Step 2:** Days under 30/360 from Jan 15 to Mar 15: $30 \times (3-1) + (15-15) = 60$ days. Year fraction = 60/360 = 0.1667.

**Step 3:** Accrued interest: $1{,}000{,}000 \times 0.06 \times 0.1667 = \$10{,}000$.

> **CFA Exam Tip:** Key associations: US Treasuries use ACT/ACT, US corporate bonds use 30/360, money markets use ACT/360.

> **Common Mistake:** Confusing 30/360 with ACT/360. Both have 360 in the denominator, but 30/360 uses a *fictional* day count in the numerator (30-day months), while ACT/360 uses *actual* days.### Which Markets Use Which Convention?

| Convention | Markets / Instruments | Logic |
|:-----------|:---------------------|:------|
| **30/360** | US corporate bonds, agency bonds | Simplifies coupon calculations (each semi-annual period = 180 days) |
| **ACT/360** | Money markets (LIBOR, SOFR), Euro commercial paper | Actual days, but 360-day year (boosts effective rate slightly) |
| **ACT/365** | UK government bonds (gilts), AUD money market | Actual days, 365-day year |
| **ACT/ACT** | US Treasury bonds, Euro government bonds | Most "accurate" — actual days in both numerator and denominator |

> **Common Mistake:** The difference between ACT/360 and ACT/365 seems trivial, but on a \$100 million position accruing for 90 days at 5%, the difference is:
> - ACT/360: $100M \times 0.05 \times 90/360 = \$1,250,000$
> - ACT/365: $100M \times 0.05 \times 90/365 = \$1,232,877$
> - Difference: \$17,123

That's real money! Day count conventions matter.

> **CFA Exam Tip:** You won't be asked to memorise which markets use which conventions, but you should understand that the convention affects the accrued interest calculation and therefore the "dirty" (full) price of a bond.


The code below implements each day count convention and compares them across various accrual periods.> **What to watch for:** The differences are largest for months with unusual lengths (February) and for periods that straddle month boundaries. The 30/360 convention will always give exactly 30 days per month, while ACT methods use the actual calendar.


In [ ]:
from datetime import date, timedelta

def days_30_360(d1, d2):
    """30/360 day count: each month = 30 days, year = 360.
    Uses the US (NASD) variant of the 30/360 rule.
    """
    y1, m1, day1 = d1.year, d1.month, min(d1.day, 30)
    y2, m2, day2 = d2.year, d2.month, d2.day
    if day2 == 31 and day1 >= 30:
        day2 = 30
    if day1 == 31:
        day1 = 30
    return 360 * (y2 - y1) + 30 * (m2 - m1) + (day2 - day1)

def year_frac_30_360(d1, d2):
    return days_30_360(d1, d2) / 360.0

def year_frac_act_360(d1, d2):
    return (d2 - d1).days / 360.0

def year_frac_act_365(d1, d2):
    return (d2 - d1).days / 365.0

def year_frac_act_act(d1, d2):
    """ACT/ACT (simplified: uses actual days / actual days in year)."""
    actual_days = (d2 - d1).days
    # Determine if period spans a leap year
    year_start = date(d1.year, 1, 1)
    year_end = date(d1.year + 1, 1, 1)
    days_in_year = (year_end - year_start).days
    return actual_days / days_in_year


# --- Compare conventions across a specific accrual period ---
d1 = date(2024, 1, 15)
d2 = date(2024, 7, 15)

print(f"Accrual period: {d1} to {d2}")
print(f"Actual days: {(d2 - d1).days}")
print()
conventions = [
    ('30/360',   year_frac_30_360),
    ('ACT/360',  year_frac_act_360),
    ('ACT/365',  year_frac_act_365),
    ('ACT/ACT',  year_frac_act_act),
]

notional = 1_000_000
coupon_rate = 0.05

print(f"Accrued interest on ${notional:,.0f} at {coupon_rate:.1%}:")
print("-" * 50)
for name, func in conventions:
    yf = func(d1, d2)
    accrued = notional * coupon_rate * yf
    print(f"  {name:<10s}  year_frac = {yf:.6f}   accrued = ${accrued:>12,.2f}")

# Compare across multiple periods (including tricky dates)
print("\n\nYear fraction comparison across different periods:")
print("-" * 75)
periods = [
    (date(2024, 1, 1), date(2024, 4, 1)),    # Q1 in a leap year
    (date(2024, 2, 1), date(2024, 3, 1)),     # Feb in leap year (29 days)
    (date(2023, 2, 1), date(2023, 3, 1)),     # Feb in non-leap year (28 days)
    (date(2024, 1, 1), date(2025, 1, 1)),     # Full leap year (366 days)
    (date(2023, 1, 1), date(2024, 1, 1)),     # Full non-leap year (365 days)
]

print(f"{'Period':<25s} {'30/360':>10s} {'ACT/360':>10s} {'ACT/365':>10s} {'ACT/ACT':>10s}")
print("-" * 75)
for d1, d2 in periods:
    fracs = [f(d1, d2) for _, f in conventions]
    print(f"  {str(d1)} to {str(d2)}  " + "  ".join(f"{x:>8.6f}" for x in fracs))

**Interpreting the output:**

The differences become most apparent in two cases:

1. **February:** Under 30/360, February is treated as having 30 days, so Feb 1 to Mar 1 always gives 30/360 = 0.0833. But actual days are 28 or 29, causing ACT-based conventions to give smaller fractions.

2. **Full year:** A full non-leap year gives exactly 1.0 under ACT/365 and ACT/ACT, but 365/360 = 1.01389 under ACT/360 — lenders earn slightly more than the quoted rate.

### Summary: When to Use What

| If you are working with... | Use this convention |
|---------------------------|---------------------|
| US Treasury bonds | ACT/ACT |
| US corporate bonds | 30/360 |
| SOFR/LIBOR-based instruments | ACT/360 |
| UK gilts | ACT/365 |
| Interest rate swaps (USD) | ACT/360 (floating leg), 30/360 (fixed leg) |
| Euro government bonds | ACT/ACT |> **Key Concept:** Day count conventions are a source of real pricing differences in financial markets. When you see a bond yield quoted as "5.00%," the actual cash flow depends on WHICH 5% — ACT/ACT 5% produces different cash flows from 30/360 5%. This is why settlement systems must agree on conventions.

### Accrued Interest and Day Counts

When you buy a bond between coupon dates, you owe the seller **accrued interest** — the interest earned since the last coupon. The day count convention determines exactly how this is calculated:

$$\text{Accrued Interest} = \text{Coupon} \times \frac{\text{Days since last coupon}}{\text{Days in coupon period}}$$

The "days" in both numerator and denominator depend on the convention used.


## 10. Summary: The Interest Rate Toolkit

We have built the complete foundation for understanding interest rates:

| Concept | Key Formula | When to Use |
|---------|-----------|-------------|
| Simple Interest | $A = P(1 + rt)$ | Short-term money market instruments |
| Compound Interest | $A = P(1 + r/m)^{mt}$ | Bonds, loans, most financial products |
| Continuous Compounding | $A = Pe^{rt}$ | Derivatives pricing, theoretical models |
| Discount Factor | $d(t) = e^{-rt}$ | Valuing any future cash flow |
| Forward Rate | $f(t_1,t_2) = \frac{r_2 t_2 - r_1 t_1}{t_2 - t_1}$ | Rate expectations, hedging |
| EAR | $(1+r/m)^m - 1$ | Comparing investments |

### The Big Picture

Everything in this notebook is connected by a single thread: **the time value of money**. Simple interest is the first-order approximation. Compound interest is the realistic model. Continuous compounding is the mathematical limit. Rate conversions translate between languages. Discount factors apply the concept to valuation. Forward rates extend it to future periods. Day-count conventions ensure precision in real-world calculations.

With these tools in hand, you are ready to tackle bond pricing, duration, yield curves, and the full apparatus of fixed-income analysis.### CFA Level 1 Curriculum Alignment

Key Learning Outcome Statements covered:
- LOS: Calculate and interpret effective annual rate given stated annual rate and compounding frequency
- LOS: Calculate the future value and present value of a single sum of money
- LOS: Describe and compare different yield measures (BEY, EAR, money market yield)
- LOS: Describe spot rates, forward rates, and their relationships
- LOS: Describe common day-count conventions

### Formula Reference Card

| Concept | Formula |
|:---|:---|
| Simple interest | $FV = PV(1 + rt)$ |
| Compound interest | $FV = PV(1 + r/m)^{mt}$ |
| Continuous compounding | $FV = PV \cdot e^{rt}$ |
| EAR | $(1 + r_{\text{nom}}/m)^m - 1$ |
| Nominal → Continuous | $r_c = m\ln(1 + r_{\text{nom}}/m)$ |
| Discount factor | $d(t) = e^{-r_c t}$ or $(1+r)^{-t}$ |
| Forward rate | $f(t_1, t_2) = \frac{(1+s_2)^{t_2}/(1+s_1)^{t_1}}{1}^{1/(t_2-t_1)} - 1$ |


## 11. References

1. Hull, J. C. *Options, Futures, and Other Derivatives*, 11th ed. Pearson, 2022. — Chapters 4-6.
2. Tuckman, B. & Serrat, A. *Fixed Income Securities*, 3rd ed. Wiley, 2011. — Chapters 1-3.
3. Fabozzi, F. J. *Fixed Income Mathematics*, 4th ed. McGraw-Hill, 2006.
4. Brigo, D. & Mercurio, F. *Interest Rate Models — Theory and Practice*, 2nd ed. Springer, 2006. — Chapter 1.
5. CFA Institute. *CFA Program Curriculum Level I*, "Quantitative Methods" and "Fixed Income" volumes.### Further Reading for CFA Candidates

- CFA Institute. *CFA Program Curriculum Level I*, Quantitative Methods: The Time Value of Money — covers simple/compound interest, EAR, and rate conversions in the exam context
- Tuckman, B. & Serrat, A. *Fixed Income Securities*, Chapter 1 — excellent treatment of discount factors and forward rates from a practitioner's perspective


> **Final note:** Mastery of the concepts in this notebook is essential for the CFA Level 1 exam, as well as for practical financial analysis work. Practice the worked examples by hand and verify your understanding by reproducing the code from scratch.
